# 03 RFM 用户分层分析

基于 Recency（最近购买）、Frequency（购买频率）、Monetary（消费金额）三维度对用户进行价值分层，输出运营策略建议。

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

from pathlib import Path
BASE_DIR = Path('..').resolve()

## 3.1 加载用户特征数据

In [ ]:
# 加载用户特征数据
data_path = BASE_DIR / 'data' / 'processed' / 'user_features.parquet'
df = pd.read_parquet(data_path)
print(f"数据加载完成，共 {len(df)} 位用户")
print(f"\n可用列: {df.columns.tolist()}")
df.head()

## 3.2 RFM 各维度分布

- **R (Recency)**: 最近购买距今天数 — 值越小越活跃
- **F (Frequency)**: 购买频次 — 值越大越忠诚
- **M (Monetary)**: 累计消费金额 — 值越大越有价值

In [ ]:
# RFM 维度分布
rfm_cols = {
    'recency_days': 'R - 最近购买距今天数',
    'total_orders': 'F - 购买频次',
    'total_spend': 'M - 累计消费金额'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col, title) in zip(axes, rfm_cols.items()):
    if col not in df.columns:
        ax.set_title(f'{title}\n(列不存在)')
        continue
    data = df[col].dropna()
    ax.hist(data, bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.5,
               label=f'中位数: {data.median():.1f}')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(col)
    ax.set_ylabel('用户数')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('RFM 各维度分布', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# 基本统计
for col, title in rfm_cols.items():
    if col in df.columns:
        print(f"\n{title}:")
        print(df[col].describe())

## 3.3 RFM 分层规则

使用四分位数将每个维度分为高/低两档，组合形成 RFM 分层标签。

In [ ]:
# RFM 分层总览
segment_col = None
for col in ['rfm_segment', 'rfm_label', 'customer_segment']:
    if col in df.columns:
        segment_col = col
        break

if segment_col:
    print(f"使用分层列: {segment_col}")
    segment_counts = df[segment_col].value_counts()
    print(f"\n各层级用户分布:")
    print(segment_counts)
else:
    print("未找到 RFM 分层列，将基于 RFM 维度手动分层")
    # 手动 RFM 分层
    if all(c in df.columns for c in ['recency_days', 'total_orders', 'total_spend']):
        df['R_score'] = pd.qcut(df['recency_days'], q=4, labels=[4, 3, 2, 1])
        df['F_score'] = pd.qcut(df['total_orders'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
        df['M_score'] = pd.qcut(df['total_spend'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
        df['RFM_total'] = df['R_score'].astype(int) + df['F_score'].astype(int) + df['M_score'].astype(int)
        
        def rfm_label(score):
            if score >= 10:
                return '重要价值客户'
            elif score >= 8:
                return '重要发展客户'
            elif score >= 6:
                return '一般价值客户'
            else:
                return '流失客户'
        
        df['rfm_segment'] = df['RFM_total'].apply(rfm_label)
        segment_col = 'rfm_segment'
        segment_counts = df[segment_col].value_counts()
        print("手动 RFM 分层完成:")
        print(segment_counts)

## 3.4 RFM 分层可视化

In [ ]:
# RFM 分层可视化
if segment_col:
    segment_counts = df[segment_col].value_counts()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 饼图
    colors = plt.cm.Set3(np.linspace(0, 1, len(segment_counts)))
    axes[0].pie(segment_counts.values, labels=segment_counts.index,
                autopct='%1.1f%%', colors=colors, startangle=90,
                textprops={'fontsize': 10})
    axes[0].set_title('RFM 用户分层占比', fontsize=13)
    
    # 柱状图
    sns.barplot(x=segment_counts.index, y=segment_counts.values, ax=axes[1], palette='Set2')
    axes[1].set_title('各层级用户数量', fontsize=13)
    axes[1].set_xlabel('用户层级')
    axes[1].set_ylabel('用户数')
    axes[1].tick_params(axis='x', rotation=45)
    for i, v in enumerate(segment_counts.values):
        axes[1].text(i, v + max(segment_counts.values) * 0.01, str(v), ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## 3.5 各层级用户画像

In [ ]:
# 各层级用户画像统计表
if segment_col:
    agg_dict = {}
    if 'total_spend' in df.columns:
        agg_dict['total_spend'] = 'mean'
    if 'total_orders' in df.columns:
        agg_dict['total_orders'] = 'mean'
    if 'recency_days' in df.columns:
        agg_dict['recency_days'] = 'mean'
    if 'is_churned' in df.columns:
        agg_dict['is_churned'] = 'mean'

    profile = df.groupby(segment_col).agg(agg_dict)
    profile['用户数'] = df.groupby(segment_col).size()
    profile['用户占比'] = profile['用户数'] / len(df)

    rename_map = {
        'total_spend': '平均消费金额',
        'total_orders': '平均购买频次',
        'recency_days': '平均最近购买天数',
        'is_churned': '流失率'
    }
    profile = profile.rename(columns=rename_map)
    print("各层级用户画像统计表:")
    profile.round(2)

## 3.6 层级间 KPI 对比

In [ ]:
# 层级间 KPI 对比（箱线图）
if segment_col:
    kpi_cols = {
        'total_spend': '累计消费金额',
        'total_orders': '购买频次',
        'recency_days': '最近购买天数'
    }
    available_kpis = {k: v for k, v in kpi_cols.items() if k in df.columns}
    
    n_kpis = len(available_kpis)
    fig, axes = plt.subplots(1, n_kpis, figsize=(5 * n_kpis, 6))
    if n_kpis == 1:
        axes = [axes]
    
    for ax, (col, label) in zip(axes, available_kpis.items()):
        order = df.groupby(segment_col)[col].mean().sort_values(ascending=False).index
        sns.boxplot(data=df, x=segment_col, y=col, ax=ax, order=order, palette='Set2')
        ax.set_title(f'各层级 {label} 对比', fontsize=12)
        ax.set_xlabel('')
        ax.set_ylabel(label)
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('RFM 层级间 KPI 对比', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## 3.7 运营策略建议

| 层级 | 特征 | 运营策略 |
|------|------|----------|
| 重要价值客户 | R低F高M高 | VIP专属服务、新品优先体验 |
| 重要发展客户 | R低F低M高 | 关联推荐提升频次、满减刺激复购 |
| 重要保持客户 | R高F高M高 | 大额优惠券召回、流失原因调研 |
| 一般价值客户 | 各维度一般 | 新手引导、小额满减培养习惯 |
| 流失客户 | R高F低M低 | 低成本自动化触达、大促批量召回 |

## 小结

- RFM 分层将用户分为高/中/低价值群体
- 高价值用户占比较小但贡献了主要收入
- 不同层级用户需要差异化运营策略
- 重点关注「重要保持客户」的召回和「重要发展客户」的频次提升